In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\User\Desktop\Data\metadata.csv")

df.columns

cols_to_check = [
    "fixed_point_x",
    "fixed_point_y",
    "force_point_x",
    "force_point_y",
    "force_direction_degrees"
]

duplicates = df.loc[
    df.duplicated(subset=cols_to_check, keep="first"),
    "filename"
].tolist()

print(duplicates)



[]


In [3]:
df.head()

,serial_number,filename,fixed_point_x,fixed_point_y,force_point_x,force_point_y,force_direction_degrees
0,1,1.png,5,1,10,6,180
1,2,2.png,8,9,10,1,180
2,3,3.png,4,0,1,10,180
3,4,4.png,0,1,10,9,270
4,5,5.png,5,0,10,8,270


In [4]:
from pathlib import Path

metadata_path = Path(r"C:\Users\User\Desktop\Data\metadata.csv")
data_dir = metadata_path.parent

# Backup original metadata first
backup_path = metadata_path.with_name("metadata_backup.csv")
df.to_csv(backup_path, index=False)

# Delete PNG files whose filenames are in duplicates
for filename in duplicates:
    png_path = data_dir / filename

    if png_path.exists():
        png_path.unlink()
        print(f"Deleted image: {png_path}")
    else:
        print(f"Image not found: {png_path}")

# Remove rows from dataframe where filename is in duplicates
df = df[~df["filename"].isin(duplicates)]

# Save cleaned dataframe back to metadata.csv
df.to_csv(metadata_path, index=False)

print("Deleted duplicate rows and PNG files.")
print(f"Backup saved at: {backup_path}")

Deleted duplicate rows and PNG files.
Backup saved at: C:\Users\User\Desktop\Data\metadata_backup.csv


In [6]:
from PIL import Image

metadata_path = Path(r"C:\Users\User\Desktop\Data\metadata.csv")
data_dir = metadata_path.parent

df = pd.read_csv(metadata_path)

# Backup metadata before appending new rows
backup_path = metadata_path.with_name("metadata_before_diagonal_flip_backup.csv")
df.to_csv(backup_path, index=False)

# Use only PNGs numbered from 0 to 500
source_df = df[
    df["filename"].apply(
        lambda name: Path(str(name)).stem.isdigit()
        and 0 <= int(Path(str(name)).stem) <= 500
    )
].copy()

# New files start after the highest existing numbered PNG
existing_numbers = [
    int(p.stem)
    for p in data_dir.glob("*.png")
    if p.stem.isdigit()
]

next_number = max(existing_numbers) + 1


def flip_point_diagonally(x, y):
    """
    Diagonal flip across the line y = x.

    Old point: (x, y)
    New point: (y, x)
    """
    return y, x


def flip_force_direction_diagonally(direction_degrees):
    """
    Diagonal flip across y = x.

    0   -> 90
    90  -> 0
    180 -> 270
    270 -> 180
    """
    mapping = {
        0: 90,
        90: 0,
        180: 270,
        270: 180
    }
    return mapping[int(direction_degrees)]


new_rows = []

for _, row in source_df.iterrows():
    old_filename = row["filename"]
    old_image_path = data_dir / old_filename

    if not old_image_path.exists():
        print(f"Skipping missing image: {old_image_path}")
        continue

    new_filename = f"{next_number}.png"
    new_image_path = data_dir / new_filename

    # Diagonal flip across y = x
    # PIL TRANSPOSE swaps x and y axes
    image = Image.open(old_image_path)
    flipped_image = image.transpose(Image.Transpose.TRANSPOSE)
    flipped_image.save(new_image_path)

    # Flip fixed point 1 & 2
    fpx, fpy = flip_point_diagonally(
        int(row["fixed_point_x"]),
        int(row["fixed_point_y"])
    )

    # Flip force point
    force_x, force_y = flip_point_diagonally(
        int(row["force_point_x"]),
        int(row["force_point_y"])
    )

    # Flip force direction
    new_force_direction = flip_force_direction_diagonally(
        int(row["force_direction_degrees"])
    )

    new_row = row.copy()

    if "serial_number" in new_row.index:
        new_row["serial_number"] = next_number

    new_row["filename"] = new_filename

    new_row["fixed_point_x"] = fpx
    new_row["fixed_point_y"] = fpy


    new_row["force_point_x"] = force_x
    new_row["force_point_y"] = force_y
    new_row["force_direction_degrees"] = new_force_direction

    new_rows.append(new_row)

    print(f"Created {new_filename} from {old_filename}")

    next_number += 1


new_df = pd.DataFrame(new_rows)
df_updated = pd.concat([df, new_df], ignore_index=True)

df_updated.to_csv(metadata_path, index=False)

print(f"Added {len(new_rows)} diagonally flipped images.")
print(f"Updated metadata file: {metadata_path}")
print(f"Backup saved at: {backup_path}")

Created 501.png from 1.png
Created 502.png from 2.png
Created 503.png from 3.png
Created 504.png from 4.png
Created 505.png from 5.png
Created 506.png from 6.png
Created 507.png from 7.png
Created 508.png from 8.png
Created 509.png from 9.png
Created 510.png from 10.png
Created 511.png from 11.png
Created 512.png from 12.png
Created 513.png from 13.png
Created 514.png from 14.png
Created 515.png from 15.png
Created 516.png from 16.png
Created 517.png from 17.png
Created 518.png from 18.png
Created 519.png from 19.png
Created 520.png from 20.png
Created 521.png from 21.png
Created 522.png from 22.png
Created 523.png from 23.png
Created 524.png from 24.png
Created 525.png from 25.png
Created 526.png from 26.png
Created 527.png from 27.png
Created 528.png from 28.png
Created 529.png from 29.png
Created 530.png from 30.png
Created 531.png from 31.png
Created 532.png from 32.png
Created 533.png from 33.png
Created 534.png from 34.png
Created 535.png from 35.png
Created 536.png from 36.png
C

In [10]:
import numpy as np


data_dir = Path(r"C:\Users\User\Desktop\Data")
output_path = data_dir / "Images.npy"

png_files = sorted(
    [p for p in data_dir.glob("*.png") if p.stem.isdigit()],
    key=lambda p: int(p.stem)
)

rows = []

for png_path in png_files:
    image = Image.open(png_path).convert("L")  # grayscale

    if image.size != (256, 256):
        raise ValueError(f"{png_path.name} is {image.size}, expected 256x256")

    pixels = np.asarray(image, dtype=np.uint8).reshape(-1)
    rows.append(pixels)

images_array = np.stack(rows, axis=0)

np.save(output_path, images_array)

print(f"Saved: {output_path}")
print(f"Shape: {images_array.shape}")
print(f"Rows: {images_array.shape[0]}")
print(f"Columns per row: {images_array.shape[1]}")

Saved: C:\Users\User\Desktop\Data\Images.npy
Shape: (968, 65536)
Rows: 968
Columns per row: 65536


In [11]:
df = pd.read_csv(r"C:\Users\User\Desktop\Data\metadata.csv")

In [12]:
columns_to_drop = [
    "serial_number",
    "filename",
]

df = df.drop(columns=columns_to_drop)

In [17]:
direction_map = {
    0: (1, 0),
    90: (0, 1),
    180: (-1, 0),
    270: (0, -1),
}

df[["direction_x", "direction_y"]] = (
    df["force_direction_degrees"]
    .map(direction_map)
    .apply(pd.Series)
)

df = df.drop(columns=["force_direction_degrees"])

In [18]:
array = df.to_numpy(dtype=np.float32)

output_path = metadata_path.with_name("metadata_encoded.npy")
np.save(output_path, array)

print("Saved:", output_path)
print("Shape:", array.shape)

Saved: C:\Users\User\Desktop\Data\metadata_encoded.npy
Shape: (968, 6)


In [ ]:
df